In [1]:
import os
import boto3
from sagemaker import get_execution_role
import shutil
from pprint import pprint
import time

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


### Constants

In [2]:
# function name
str_function_name = 'genxii-pd-concat-tuning-2'

### 1. Create container

### Create ```Dockerfile```

In [3]:
%%writefile Dockerfile

FROM public.ecr.aws/lambda/python:3.8

# update pip
RUN pip install --upgrade pip

# install dependencies from project folder
COPY requirements.txt  .
RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"

# copy function code
COPY lambda_function.py ${LAMBDA_TASK_ROOT}

# Set the CMD to your handler (could also be done as a parameter override outside of the Dockerfile)
CMD ["lambda_function.lambda_handler"] 

Writing Dockerfile


### Write ```requirements.txt```

In [4]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0
pandas==1.2.4
boto3==1.24.59

Writing requirements.txt


### Write ```lambda_function.py```

In [5]:
%%writefile lambda_function.py

import pandas as pd
import boto3

# lambda handler
def lambda_handler(event, context):
    # constants
    str_project = '20231010-gen-xii'
    str_prefix = '02_pricing_pd/02_model/02_model/02_batch_tuning/models'
    
    # get df_hyperparameters
    str_filename = 'df_hyperparameters.csv'
    str_uri = f's3://{str_project}/02_pricing_pd/02_model/02_model/12_step_function/{str_filename}'
    df = pd.read_csv(str_uri)
    # convert to dict
    dict_hyperparameters = dict(zip(df['keys'], df['values']))
    
    # get eval metric
    str_eval_metric = dict_hyperparameters['STR_EVAL_METRIC']
    print(f'Eval metric: {str_eval_metric}')
            
    # get index files in s3
    print('Getting files in s3...')
    cls_client = boto3.resource('s3')
    cls_bucket = cls_client.Bucket(str_project)
    list_str_filenames = []
    for file in cls_bucket.objects.filter(Prefix=str_prefix):
        # get key
        str_key = file.key
        # make sure it is a .csv
        if '.csv' in str_key:
            # get filename
            str_filename = str_key.split('/')[-1]
            list_str_filenames.append(str_filename)
    print(f'There are {len(list_str_filenames)} files to import')
    
    # iterate and import
    print('Importing files...')
    list_df = []
    for str_filename in list_str_filenames:
        str_uri = f's3://{str_project}/{str_prefix}/{str_filename}'
        df = pd.read_csv(str_uri)
        list_df.append(df)
    
    # create df
    print('Creating data frame...')
    df = pd.concat(list_df)
    del list_df
    
    # logic for sorting
    print('Sorting...')
    if str_eval_metric in ['AUC', 'PRAUC', 'F1']:
        bool_ascending = False
    else:
        bool_ascending = True
    df.sort_values(by='flt_eval_metric_valid', ascending=bool_ascending, inplace=True)
    
    # write to s3
    print('Writing to s3...')
    str_filename = 'df_tuning.csv'
    str_uri = f's3://{str_project}/02_pricing_pd/02_model/02_model/03_lambda_concat_tuning/{str_filename}'
    df.to_csv(str_uri, index=False)

Writing lambda_function.py


### Build image and push to ECR

In [6]:
%%sh

# name the image
image=genxii-pd-concat-tuning-2

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

Sending build context to Docker daemon  15.87kB
Step 1/6 : FROM public.ecr.aws/lambda/python:3.8
 ---> 54a294f880ad
Step 2/6 : RUN pip install --upgrade pip
 ---> Using cache
 ---> bccff1672d23
Step 3/6 : COPY requirements.txt  .
 ---> 9a1831b991e1
Step 4/6 : RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"
 ---> Running in 30415fea12ed
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 65.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.8/138.8 kB 21.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 31.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.5/132.5 kB 16.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 20.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 84.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 39.2 MB/s eta 0:00:00


Removing intermediate container 30415fea12ed
 ---> 558b779d17fc
Step 5/6 : COPY lambda_function.py ${LAMBDA_TASK_ROOT}
 ---> b104e8ac2648
Step 6/6 : CMD ["lambda_function.lambda_handler"]
 ---> Running in a2b67b41c5ac
Removing intermediate container a2b67b41c5ac
 ---> d20fa4a54378
Successfully built d20fa4a54378
Successfully tagged genxii-pd-concat-tuning-2:latest


WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded
{
    "repository": {
        "repositoryArn": "arn:aws:ecr:us-west-2:836690756591:repository/genxii-pd-concat-tuning-2",
        "registryId": "836690756591",
        "repositoryName": "genxii-pd-concat-tuning-2",
        "repositoryUri": "836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-pd-concat-tuning-2",
        "createdAt": 1697670025.0,
        "imageTagMutability": "MUTABLE",
        "imageScanningConfiguration": {
            "scanOnPush": true
        },
        "encryptionConfiguration": {
            "encryptionType": "AES256"
        }
    }
}
The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-pd-concat-tuning-2]
ec7587729278: Preparing
aeab4b6c3328: Preparing
0d549fa51922: Preparing
00e37b3ec5b7: Preparing
2ae7c19e0b5c: Preparing
9e6784b558c3: Preparing
15dd6c63f3a2: Preparing
8819b61d0672: Preparing
30349c0bf45d: Preparing
91232f425615: Preparing
9e6784b558c3: Waiting
15dd6c63f3a2: Waiting
8819b61d0672: Waiting
30349c0bf

### 2. Create lambda function from image

In [7]:
# initialize class
cls_client_lambda = boto3.client('lambda')

In [8]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml
Role: arn:aws:iam::836690756591:role/risk-ops-role


In [9]:
# delete it if it exists
try:
    dict_response = cls_client_lambda.delete_function(
        FunctionName=str_function_name,
    )
    pprint(dict_response)
except:
    pass

In [10]:
# create function
str_image_uri = '836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-pd-concat-tuning-2:latest' # this must match what we name the image above
dict_response = cls_client_lambda.create_function(
    FunctionName=str_function_name,
    Role=str_role,
    Code={
        'ImageUri': str_image_uri,
    },
    Timeout=60,
    MemorySize=512,
    Publish=True,
    PackageType='Image',
    Architectures=[
        'x86_64',
    ],
    EphemeralStorage={
        'Size': 512,
    },
)
pprint(dict_response)
time.sleep(40)

{'Architectures': ['x86_64'],
 'CodeSha256': 'd6385cd4cb7e3f3157c2ebad72397918670690adbc7a5d2bdd3bfb1db98e5f99',
 'CodeSize': 0,
 'Description': '',
 'EphemeralStorage': {'Size': 512},
 'FunctionArn': 'arn:aws:lambda:us-west-2:836690756591:function:genxii-pd-concat-tuning-2',
 'FunctionName': 'genxii-pd-concat-tuning-2',
 'LastModified': '2023-10-18T23:01:23.137+0000',
 'MemorySize': 512,
 'PackageType': 'Image',
 'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '1070',
                                      'content-type': 'application/json',
                                      'date': 'Wed, 18 Oct 2023 23:01:24 GMT',
                                      'x-amzn-requestid': '10ac9b74-fcca-452f-8c80-15b3d76e1c10'},
                      'HTTPStatusCode': 201,
                      'RequestId': '10ac9b74-fcca-452f-8c80-15b3d76e1c10',
                      'RetryAttempts': 0},
 'RevisionId': 'd6b11f47-438c-4d15-998

### Clean-up

In [11]:
for str_file in ['Dockerfile', 'lambda_function.py', 'requirements.txt']:
    os.remove(str_file)